In [21]:
import pandas as pd
import numpy as np

from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    RandomizedSearchCV
)

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report
)

from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.preprocessing import label_binarize

import mlflow
import mlflow.sklearn

import warnings
warnings.filterwarnings("ignore")

In [22]:
df = pd.read_csv("D:\Customer Support\Data set\customer_support_tickets_FE.csv")

print(df.shape)

df.head()

(8469, 25)


,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,...,Time to Resolution,Customer Satisfaction Rating,Purchase_Year,Purchase_Month,Purchase_Day,Purchase_Weekday,Description_Char_Count,Description_Word_Count,Subject_Char_Count,Subject_Word_Count
0,1,Marisa Obrien,carrollallison@example.com,32,2,16,2021-03-22,5,Product setup,I'm having an issue with the {product_purchase...,...,NaN,0.000000,2021,3,22,0,284,43,13,2
1,2,Jessica Rios,clarkeashley@example.com,42,0,21,2021-05-22,5,Peripheral compatibility,I'm having an issue with the {product_purchase...,...,NaN,0.000000,2021,5,22,5,282,44,24,2
2,3,Christopher Robbins,gonzalestracy@example.com,48,2,10,2020-07-14,5,Network problem,I'm facing a problem with my {product_purchase...,...,NaN,1.386294,2020,7,14,1,275,42,15,2
3,4,Christina Dillon,bradleyolson@example.org,27,0,25,2020-11-13,1,Account access,I'm having an issue with the {product_purchase...,...,NaN,1.386294,2020,11,13,4,262,41,14,2
4,5,Alexander Carroll,bradleymark@example.com,67,0,5,2020-02-04,1,Data loss,I'm having an issue with the {product_purchase...,...,NaN,0.693147,2020,2,4,1,333,55,9,2


In [23]:
df["text"] = (
    df["Ticket Subject"].fillna("")
    + " "
    + df["Ticket Description"].fillna("")
)

In [24]:
target_encoder = LabelEncoder()

y = target_encoder.fit_transform(
    df["Ticket Type"]
)

print("Classes:", len(np.unique(y)))

Classes: 5


In [25]:
tfidf = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95,
    stop_words="english",
    sublinear_tf=True
)

X_text = tfidf.fit_transform(df["text"])

print(X_text.shape)

(8469, 9962)


In [26]:
categorical_cols = [
    "Customer Gender",
    "Product Purchased",
    "Ticket Channel"
]

for col in categorical_cols:

    le = LabelEncoder()

    df[col] = le.fit_transform(
        df[col].astype(str)
    )

In [27]:
tabular_cols = [
    "Customer Age",
    "Customer Gender",
    "Product Purchased",
    "Ticket Channel",
    "Description_Char_Count",
    "Description_Word_Count",
    "Subject_Char_Count",
    "Subject_Word_Count"
]

X_tabular = csr_matrix(
    df[tabular_cols].fillna(0)
)

print(X_tabular.shape)

(8469, 8)


In [28]:
X = hstack([
    X_text,
    X_tabular
])

print(X.shape)

(8469, 9970)


In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [30]:
mlflow.set_tracking_uri(
    "file:./mlruns"
)

mlflow.set_experiment(
    "Advanced_Classical_ML"
)

<Experiment: artifact_location='file:///d:/Customer Support/Models/mlruns/810514993562148053', creation_time=1781659677879, experiment_id='810514993562148053', last_update_time=1781659677879, lifecycle_stage='active', name='Advanced_Classical_ML', tags={}, trace_location=None, workspace='default'>

In [31]:
def evaluate_cv(model):

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="f1_macro",
        n_jobs=-1
    )

    return scores.mean()

In [42]:
with mlflow.start_run(run_name="RandomForest"):

    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=20,
        min_samples_split=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    rf_cv = evaluate_cv(rf)

    rf.fit(X_train, y_train)

    rf_preds = rf.predict(X_test)

    rf_acc = accuracy_score(
        y_test,
        rf_preds
    )

    rf_f1 = f1_score(
        y_test,
        rf_preds,
        average="macro"
    )
    rf_probs = rf.predict_proba(X_test)

    y_test_bin = label_binarize(
    y_test,
    classes=np.unique(y)
    )
    
    rf_roc = roc_auc_score(
    y_test_bin,
    rf_probs,
    multi_class="ovr"
    )


    mlflow.log_metric("cv_f1", rf_cv)
    mlflow.log_metric("accuracy", rf_acc)
    mlflow.log_metric("f1_macro", rf_f1)
    mlflow.log_metric("roc_auc",rf_roc)

    mlflow.sklearn.log_model(
        rf,
        "random_forest"
    )

    print(
        f"Accuracy={rf_acc:.4f}, "
        f"F1={rf_f1:.4f},"
        f"ROC={rf_roc:.4f}"
    )

2026/06/19 07:44:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/19 07:44:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Accuracy=0.2007, F1=0.1997,ROC=0.4975


In [33]:
with mlflow.start_run(run_name="XGBoost"):

    xgb = XGBClassifier(
        n_estimators=300,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42
    )

    xgb_cv = evaluate_cv(xgb)

    xgb.fit(X_train, y_train)

    xgb_preds = xgb.predict(X_test)

    xgb_probs = xgb.predict_proba(X_test)

    xgb_acc = accuracy_score(
        y_test,
        xgb_preds
    )

    xgb_f1 = f1_score(
        y_test,
        xgb_preds,
        average="macro"
    )

    y_test_bin = label_binarize(
        y_test,
        classes=np.unique(y)
    )

    xgb_roc = roc_auc_score(
        y_test_bin,
        xgb_probs,
        multi_class="ovr"
    )

    mlflow.log_metric("cv_f1", xgb_cv)
    mlflow.log_metric("accuracy", xgb_acc)
    mlflow.log_metric("f1_macro", xgb_f1)
    mlflow.log_metric("roc_auc", xgb_roc)

    mlflow.sklearn.log_model(
        xgb,
        "xgboost"
    )

    print(
        f"Accuracy={xgb_acc:.4f}, "
        f"F1={xgb_f1:.4f}, "
        f"ROC={xgb_roc:.4f}"
    )

2026/06/18 13:12:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/18 13:12:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Accuracy=0.2113, F1=0.2108, ROC=0.5058


In [34]:
with mlflow.start_run(run_name="LightGBM"):

    lgbm = LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=8,
        random_state=42
    )

    lgbm_cv = evaluate_cv(lgbm)

    lgbm.fit(X_train, y_train)

    lgbm_preds = lgbm.predict(X_test)

    lgbm_probs = lgbm.predict_proba(X_test)

    lgbm_acc = accuracy_score(
        y_test,
        lgbm_preds
    )

    lgbm_f1 = f1_score(
        y_test,
        lgbm_preds,
        average="macro"
    )

    y_test_bin = label_binarize(
        y_test,
        classes=np.unique(y)
    )

    lgbm_roc = roc_auc_score(
        y_test_bin,
        lgbm_probs,
        multi_class="ovr"
    )

    mlflow.log_metric("cv_f1", lgbm_cv)
    mlflow.log_metric("accuracy", lgbm_acc)
    mlflow.log_metric("f1_macro", lgbm_f1)
    mlflow.log_metric("roc_auc", lgbm_roc)

    mlflow.sklearn.log_model(
        lgbm,
        "lightgbm"
    )

    print(
        f"Accuracy={lgbm_acc:.4f}, "
        f"F1={lgbm_f1:.4f}, "
        f"ROC={lgbm_roc:.4f}"
    )

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.045952 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 66171
[LightGBM] [Info] Number of data points in the train set: 6775, number of used features: 1034
[LightGBM] [Info] Start training from score -1.645505
[LightGBM] [Info] Start training from score -1.608700
[LightGBM] [Info] Start training from score -1.640925
[LightGBM] [Info] Start training from score -1.576053
[LightGBM] [Info] Start training from score -1.578197
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

2026/06/18 13:15:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/18 13:15:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Accuracy=0.1983, F1=0.1979, ROC=0.5007


In [35]:
param_grid = {

    "n_estimators":[200,300,500],

    "max_depth":[10,20,30,None],

    "min_samples_split":[2,5,10],

    "min_samples_leaf":[1,2,4]
}

In [36]:
rf_search = RandomizedSearchCV(

    RandomForestClassifier(),

    param_grid,

    n_iter=10,

    cv=3,

    scoring="f1_macro",

    random_state=42,

    n_jobs=-1
)

rf_search.fit(
    X_train,
    y_train
)

print(
    rf_search.best_params_
)

print(
    rf_search.best_score_
)

{'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': None}
0.20341616360036122


In [37]:
import optuna

In [38]:
def objective(trial):

    params = {

        "n_estimators":
        trial.suggest_int(
            "n_estimators",
            100,
            500
        ),

        "max_depth":
        trial.suggest_int(
            "max_depth",
            3,
            12
        ),

        "learning_rate":
        trial.suggest_float(
            "learning_rate",
            0.01,
            0.3
        ),

        "subsample":
        trial.suggest_float(
            "subsample",
            0.6,
            1.0
        ),

        "colsample_bytree":
        trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0
        ),

        "objective":"multi:softprob",

        "eval_metric":"mlogloss"
    }

    model = XGBClassifier(
        **params
    )

    score = cross_val_score(

        model,

        X_train,

        y_train,

        cv=3,

        scoring="f1_macro",

        n_jobs=-1

    ).mean()

    return score

In [39]:
study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=20
)

print(
    study.best_params
)

print(
    study.best_value
)

[I 2026-06-18 13:24:29,455] A new study created in memory with name: no-name-ce00390c-ec57-419e-a492-1f92b33f0914
[I 2026-06-18 13:36:58,510] Trial 0 finished with value: 0.2013503462914267 and parameters: {'n_estimators': 495, 'max_depth': 10, 'learning_rate': 0.28906964098639387, 'subsample': 0.7510661726743721, 'colsample_bytree': 0.8299854047674675}. Best is trial 0 with value: 0.2013503462914267.
[I 2026-06-18 13:40:53,356] Trial 1 finished with value: 0.19943087612709123 and parameters: {'n_estimators': 264, 'max_depth': 6, 'learning_rate': 0.09822436135341849, 'subsample': 0.7301682358086848, 'colsample_bytree': 0.8755460342839056}. Best is trial 0 with value: 0.2013503462914267.
[I 2026-06-18 13:44:03,136] Trial 2 finished with value: 0.19232829460352718 and parameters: {'n_estimators': 262, 'max_depth': 6, 'learning_rate': 0.21692195899881214, 'subsample': 0.8805143427883539, 'colsample_bytree': 0.723145754338636}. Best is trial 0 with value: 0.2013503462914267.
[I 2026-06-18 

{'n_estimators': 360, 'max_depth': 11, 'learning_rate': 0.2493377038989884, 'subsample': 0.791070388456006, 'colsample_bytree': 0.9931508363664502}
0.20837048279972606


In [43]:
results = pd.DataFrame({

    "Model":[
        "Random Forest",
        "XGBoost",
        "LightGBM"
    ],

    "CV_F1":[
        rf_cv,
        xgb_cv,
        lgbm_cv
    ],

    "Accuracy":[
        rf_acc,
        xgb_acc,
        lgbm_acc
    ],

    "F1_Macro":[
        rf_f1,
        xgb_f1,
        lgbm_f1
    ],

    "ROC_AUC":[
        rf_roc,
        xgb_roc,
        lgbm_roc
    ]
})

results.sort_values(
    by="F1_Macro",
    ascending=False
)

,Model,CV_F1,Accuracy,F1_Macro,ROC_AUC
1,XGBoost,0.198200,0.211334,0.210847,0.505830
0,Random Forest,0.195385,0.200708,0.199702,0.497476
2,LightGBM,0.198013,0.198347,0.197873,0.500679


In [45]:
from sklearn.svm import LinearSVC

svm = LinearSVC(
    C=1.0,
    class_weight="balanced",
    random_state=42
)

svm.fit(X_train, y_train)

preds = svm.predict(X_test)

acc = accuracy_score(y_test, preds)

f1 = f1_score(
    y_test,
    preds,
    average="macro"
)

print(acc, f1)

0.21192443919716647 0.14344004026897997


In [48]:
print(df["Ticket Type"].value_counts())

Ticket Type
4    1752
5    1747
2    1695
3    1641
1    1634
Name: count, dtype: int64


In [49]:
print(df["Ticket Type"].value_counts())

pd.crosstab(
    df["Ticket Subject"],
    df["Ticket Type"]
)

Ticket Type
4    1752
5    1747
2    1695
3    1641
1    1634
Name: count, dtype: int64


Ticket Type,1,2,3,4,5
Ticket Subject,,,,,
Account access,103,92,107,108,99
Battery life,106,104,101,119,112
Cancellation request,82,103,85,109,108
Data loss,89,115,91,97,99
Delivery problem,115,114,109,107,116
Display issue,91,103,80,99,105
Hardware issue,100,109,106,129,103
Installation support,108,99,99,119,105
Network problem,95,102,113,107,122
